# Computational notebook: docking screen of the SidA ornithine site, with a redock control and direct clash measurement

| | |
|---|---|
| **Project** | Structure-based screening of literature ligands against *Aspergillus fumigatus* SidA (L-ornithine N<sup>5</sup>-monooxygenase) |
| **Target site** | The L-ornithine site (substrate site), with FAD and NADP(H) in place |
| **Prepared by** | Leonardo Mariño-Ramírez |
| **Entry type** | Reusable protocol: change the parameters in section 2 and re-run; each run records its own date, settings and tool versions |
| **Companion files** | `sidacore/screen.py` (code), `reference_results.csv` (reference run), `README.md` (installation) |

> **How to use this entry.** Sections *Objective* to *Methods* describe what is done and why. Sections **1-9** are the procedure: they run the analysis and show the results.
> *Observations* is for the experimenter's own notes. Numbers in square brackets are references, listed at the end.

## Objective

To decide, for a list of ligands taken from the literature, whether each can be placed in the substrate site of a SidA crystal structure **without overlapping the protein**, and to
say how far the docking protocol can be trusted on that structure before any ligand is judged. The tool is for *screening by fit*; it does **not** rank binding affinity.

The worked example docks the substrate L-ornithine, two close analogues (L-lysine and L-arginine) and one deliberately oversized molecule into three SidA structures.

## Background

**The pathogen and the target.** *Aspergillus fumigatus* is a soil mould and a leading cause of invasive mould infection in immunocompromised patients [1,3]. To obtain iron it makes
hydroxamate siderophores, and the first committed step of their synthesis is the N<sup>5</sup>-hydroxylation of L-ornithine by the flavin-dependent monooxygenase SidA [3].
Deleting *sidA* abolishes production of the siderophores triacetylfusarinine C and ferricrocin, and the mutant cannot grow in low-iron medium or serum unless a siderophore is supplied [3], and SidA is essential for virulence in a murine model of
invasive aspergillosis [2]. Mammals have no *sidA* ortholog and no fungal siderophore system, which makes the pathway a candidate antifungal target [2,4].

**The enzyme.** SidA is a class B flavin-dependent monooxygenase [6,12]. It uses NADPH and O<sub>2</sub> to form a C4a-hydroperoxyflavin that hydroxylates the side-chain amine of L-ornithine [5,8].
NADP<sup>+</sup> stays bound after flavin reduction, is the last product released, and is needed to stabilise the hydroperoxyflavin [8]. SidA is highly specific for L-ornithine [5]; L-lysine also binds,
but hydroxylation is inefficient and H<sub>2</sub>O<sub>2</sub> is the main product [10].

**Structures.** Crystal structures with L-ornithine, L-lysine and L-arginine bound (with NADP<sup>+</sup>) are available [7], and structures of oxidised and reduced SidA show the FAD in an *out* or *in*
conformation: reduction of the flavin, or binding of NADP(H), is enough to drive the FAD in [11]. In the ornithine site, Lys107 forms an ionic interaction with the carboxylate of ornithine, and Asn293, Asn323
and Ser469 hydrogen-bond it; K107A abolishes activity, and Asn323 also hydrogen-bonds the nicotinamide ribose of NADP(H) [9]. Gln102 interacts with NADPH [10], and Ser257 acts as a pivot for the nicotinamide of NADP<sup>+</sup> [8].
Molecular dynamics of NADPH and L-ornithine binding has been reported [12], and the protonation state of the ornithine side-chain amine (argued to bind neutral) is still discussed [10].

**Why a control is part of the protocol.** Docking was used to explore the SidA active site in the out conformation [11]. In a PubMed Central search made for this project in November 2026 (`reports/literature_sida_docking.md`)
we found no published redocking of a crystallographic SidA ligand; that is a search result, not proof that none exists. We therefore redock the deposited ornithine into each structure first. In our own
tests this is not a formality: the deposited ornithine is recovered in some SidA structures and not in others (section 5 and the worked example below).

## Methods

### Materials: software

| Step | Program (version) | Reference |
|---|---|---|
| Structure retrieval and selection | RCSB PDB mmCIF files [13]; Biotite 1.6.0 | [13,22] |
| Protein hydrogens, protonation at a given pH | PDB2PQR 3.7.1 with PROPKA 3.5.1 (AMBER force field) | [14,15,16] |
| Receptor and ligand conversion to PDBQT | Meeko 0.8.0 (with shipped templates for FAD, FDA and NADP) | [21] |
| Ligand protonation at a given pH | Open Babel 3.2.1 | [19] |
| Ligand 3-D structure, RMSD | RDKit 2026.03.1 | [20] |
| Docking | AutoDock Vina 1.2.7, CPU | [17,18] |
| Clash radii | Bondi van der Waals radii | [23] |

The versions actually used by a run are printed in the *run record* below and saved in `run.json`.

### Materials: structures

| PDB | Content | Resolution | Space group | Used here | Source |
|---|---|---|---|---|---|
| 4B63 | SidA, oxidised FAD, NADP, L-ornithine | 1.90 Å | I 2 2 2 | yes | [7] |
| 6X0J | SidA, **reduced** FAD (in), NADP, L-ornithine; four chains (A-D) | 2.335 Å | P 1 2<sub>1</sub> 1 | yes (chains A and C) | [11] |
| 4B64, 4B66 | SidA with NADP and L-lysine (4B64) or L-arginine (4B66, reduced) | 2.28 Å, 2.9 Å | I 2 2 2 | not used; evidence that Lys and Arg bind | [7] |
| 6X0H, 6X0I | oxidised SidA, FAD out (6X0H) or FAD in with NADP (6X0I) | 2.087 Å, 1.95 Å | P 1 2<sub>1</sub> 1 | not used here | [11] |
| 5CKU | N323A mutant with NADP and L-ornithine | 2.1 Å | I 2 2 2 | not used | [9] |

(Titles, resolutions and space groups from the RCSB data API, 2026-11-05.)

### Receptor preparation (per target, section 3 shows the result)

1. The mmCIF file is read (first model). Alternative conformers are resolved by taking, for each residue, the conformer of highest occupancy.
2. One chain is the receptor. Its standard amino acids are kept. **FAD (or its reduced form FDA) and NADP are kept** at their deposited coordinates when present (`keep="auto"`), because they line the ornithine site;
   waters, crystallisation additives (sulfate, glycerol), other heterogens and non-standard residues are removed, and every removed residue type is listed with its distance to the site.
3. Hydrogens are added to the protein with PDB2PQR (AMBER force field, PROPKA pK<sub>a</sub> estimates at the chosen pH, default 7.4). Cofactor protonation comes from the Meeko templates and does not change with pH.
4. Meeko writes the receptor PDBQT. Residues Meeko cannot parameterise within 8 Å of the box centre are deleted by Meeko and reported in a note.
5. The **docking box** is a cube (default 12 Å edge) centred on the centroid of the heavy atoms of the deposited *site ligand* (L-ornithine in the example). A 12 Å box is site-sized: a ligand that does not fit is forced
   against the protein. A 22 Å box leaves room at the rim of the site.

### Ligand preparation

SMILES are protonated at the chosen pH (default 7.4) with Open Babel, embedded in 3-D with RDKit (random seed 0) and optimised with MMFF94, then converted to PDBQT with Meeko. Undefined stereocentres and ligands with more than
10 rotatable bonds are flagged. A SMILES that cannot be prepared is reported as a failed row and does not stop the others.

### Docking

AutoDock Vina 1.2.7 (CPU, its default scoring function), rigid receptor, flexible ligand, exhaustiveness 16, up to 20 poses per run, seeds 1-3 (each seed is an independent run). Vina returns fewer poses
when the remainder fall outside its energy window, so `n_poses` varies. The reported pose of a ligand is its **lowest-scoring pose over all seeds**.

### Redock control (per target)

The deposited site ligand is rebuilt from its PDB component definition, protonated and prepared exactly like the other ligands, and docked into its own receptor with the same box and settings, using five seeds.
The heavy-atom RMSD of each pose to the deposited pose is computed with RDKit, symmetry-corrected (equivalent atoms such as the two carboxylate oxygens may swap) and **without re-alignment**, since both are in the receptor frame.
**Rule (fixed beforehand in our SidA benchmark):** the target is *validated* if the **median over seeds of the top-ranked pose's RMSD is ≤ 2.0 Å**; otherwise it is marked **FAILED**. The number of seeds within 2 Å and the best of the top five poses are reported alongside.
The control uses the same box as the screen, so a tight box gives it a head start (see section 5).

### Clash measurement

For the reported pose, every heavy-atom pair between the ligand and the receptor (protein and kept cofactors) is examined. A pair is a **clash** if its distance is smaller than the sum of the Bondi van der Waals radii [23] minus a tolerance of
**0.4 Å**. Pairs of N or O atoms (potential hydrogen bonds) are instead judged by a minimum distance of **2.4 Å**. Metals and hydrogens are never counted. This is in the spirit of the all-atom contact analysis of MolProbity [24], but uses
Bondi radii and heavy atoms only. The two thresholds were set from the five deposited ornithine poses in the SidA structures (worst non-polar overlap 0.16 Å, closest N/O pair 2.50 Å), a small calibration set; the deposited ornithine shows no clashes under them.
`n_clash_free` counts how many of all docked poses have no clash.

### Is the pose in the site?

A clash-free pose can simply sit at the rim of the box. Two numbers say whether the reported pose is in the site of the deposited ligand: `centroid_dist` (distance between the centroids of the pose and the deposited ligand) and
`site_contact_fraction` (the share of the receptor residues within 4.5 Å of the deposited ligand that are also within 4.5 Å of the pose).

### The `reading` column

Fixed rules, applied in this order: no clash-free pose → *clashes in every pose*; the reported pose clashes → *best pose clashes*; clash-free but `site_contact_fraction` below 0.6 → *clash-free but outside the site*; otherwise *fits*.
A target that failed its control gets the prefix *(target failed its control)*. Vina scores are shown but are not used in the reading, because interaction-energy scores are biased towards larger compounds [25], and because in our SidA tests a wrong pose can outscore the deposited one.

### Reproducibility

Vina is deterministic for a given seed on one machine. Each run writes `results.csv`, `controls.csv`, `run.json` (settings, tool versions, ligand SMILES as docked), the best pose of each pair, and the prepared receptors.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

try:
    import sidacore
except ImportError:                       # running from a clone that was not pip-installed
    sys.path.insert(0, str(Path.cwd().parent))
    import sidacore

import pandas as pd
from sidacore import screen as S

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_columns", 30)

tools = S.check_tools()
missing = [k for k, v in tools.items() if v is None]
assert not missing, "missing programs: %s. Create the environment from environment.yml (see README.md)." % missing
print("programs found:", {k: Path(v).name for k, v in tools.items()})

## 2. Your parameters  (edit this cell, then run everything below)

**Targets.** One `TargetSpec` per structure:

| field | meaning |
|---|---|
| `pdb_id`, `chain` | the structure and the chain to use as the receptor |
| `site_ligand` | residue name of the ligand deposited in the site you care about. It centres the box and is redocked as a **control** |
| `keep` | `"auto"` keeps FAD / FDA / NADP when present (they line the SidA site); or a list, e.g. `["FAD"]` |
| `box_size` | edge of the search cube in Å. **12 = site-sized**: a ligand that does not fit is forced to clash. **22 = roomy**: an oversized ligand can dock clash-free at the rim, which is *not* a fit |
| `ph` | protonation of the protein (PROPKA) |

Waters, sulfate, glycerol and other crystallisation additives are always removed; section 3 shows exactly what.

**Ligands.** A dict `{name: SMILES}`, or the path of a `.csv` (columns `name,smiles`) or `.smi` file. Write stereochemistry
(`[C@H]`) when it matters; the notebook warns when it is missing.

In [ ]:
# ---- experimenter ----------------------------------------------------------------------------------------
EXPERIMENTER = ""           # your name; shown in the run record below

# ---- targets -------------------------------------------------------------------------------------------
TARGETS = [
    S.TargetSpec("4B63", "A", site_ligand="ORN"),   # wild type, oxidised FAD, NADP+ and L-ornithine, 1.90 A
    S.TargetSpec("6X0J", "A", site_ligand="ORN"),   # reduced FAD (FDA), NADP and L-ornithine, chain A
    S.TargetSpec("6X0J", "C", site_ligand="ORN"),   # same structure, chain C
]

# ---- ligands -------------------------------------------------------------------------------------------
LIGANDS = {
    "L-ornithine (substrate)":            "NCCC[C@H](N)C(=O)O",
    "L-lysine":                           "NCCCC[C@H](N)C(=O)O",
    "L-arginine":                         "NC(N)=NCCC[C@H](N)C(=O)O",
    "tetraphenylmethane (too big on purpose)": "C(c1ccccc1)(c1ccccc1)(c1ccccc1)c1ccccc1",
}
# LIGANDS = "my_ligands.csv"        # <- or a file from a paper: columns name,smiles

# ---- settings ------------------------------------------------------------------------------------------
SETTINGS = S.Settings(
    ligand_ph=7.4,              # ligands are protonated at this pH (None = use the SMILES exactly as written)
    seeds=(1, 2, 3),            # docking repeats per ligand; more seeds = more reliable, slower
    exhaustiveness=16,          # Vina search effort
    num_modes=20,               # poses kept per run
    control_seeds=(1, 2, 3, 4, 5),   # repeats for the redock control of each target
)

# ---- where things go -----------------------------------------------------------------------------------
RUN_NAME = "sida_example"
WORK_DIR = Path("screen_runs") / RUN_NAME      # results.csv, controls.csv, poses/, prepared receptors
CACHE_DIR = Path("structures_cache")           # downloaded structures (reused)

### Run record

Date, experimenter, tools and parameters of this entry. Keep it with the results.

In [ ]:
import datetime
import platform

print("Entry date      :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))
print("Experimenter    :", EXPERIMENTER or "(not filled in)")
print("Run name        :", RUN_NAME)
print("Machine         :", platform.platform())
print("Tools           :", S.tool_versions())
print("Targets         :", [t.name for t in TARGETS])
print("Ligands         :", len(S.load_ligands(LIGANDS)))
print("Settings        :", SETTINGS)

## 3. What will be kept and removed

Check this before docking. **Cofactors that line the site must be kept**: removing FAD or NADP changes the pocket
(in SidA, NADP's ribose is 3.3 Å from the bound ornithine). Anything listed under *removed* is gone from the receptor.

In [ ]:
for spec in TARGETS:
    sel = S.preview_target(spec, CACHE_DIR)
    print("%s   box centre %s   cofactors kept: %s" % (spec.name, [round(float(x), 1) for x in sel.center], sel.kept_cofactors))
    print(pd.DataFrame(sel.removed).rename(columns={"min_dist_to_site": "closest to site (A)"}).to_string(index=False))
    for note in sel.notes:
        print("  NOTE:", note)
    print()

## 4. Run

For each target: prepare the receptor, **redock its own deposited ligand as a control**, then dock every ligand and measure clashes.
With the example (3 targets, 4 ligands) this takes a few minutes; the receptor preparation is cached, so a re-run is faster.

In [ ]:
import time
t0 = time.time()
result = S.run_screen(TARGETS, LIGANDS, settings=SETTINGS, cache_dir=CACHE_DIR, work_dir=WORK_DIR)
print("done in %.0f s; files in %s" % (time.time() - t0, WORK_DIR.resolve()))
for name, t in result.targets.items():
    for note in t.notes:
        print("NOTE %s: %s" % (name, note))

## 5. Can the protocol be trusted on each target?  (the control)

The deposited ligand is redocked into its own structure. If the protocol cannot put it back (**median top-1 RMSD above 2 Å**),
the target is marked `FAILED` and every result against it describes the *model*, not the molecule.
`crystal_ligand_clashes` is the clash count of the deposited pose itself; it should be 0.

**A pass is a modest statement.** The control uses the same box as your screen, and that box is centred on the deposited ligand, so the protocol is
told roughly where to look. A tighter box makes the redock easier: in our SidA runs 6X0J chain C fails with a 22 Å box (median top-1 RMSD 5.96 Å) but passes with
the 12 Å box used here (about 1 Å), while 4B63 fails at every box size from 12 to 22 Å. Read *validated* as "recovers the ligand when pointed at the site", not more.

In [ ]:
result.controls

## 6. Results

In [ ]:
cols = ["target", "ligand", "reading", "best_score", "n_clashes", "worst_clash", "n_clash_free", "n_poses",
        "centroid_dist", "site_contact_fraction", "net_charge", "warnings", "status"]
shown = result.results[[c for c in cols if c in result.results.columns]]
shown.round(2)

### How to read the results

* **`reading`** is the whole table in one phrase, by fixed rules: *clashes in every pose* (no docked pose is clash-free), *best pose clashes*,
  *clash-free but outside the site*, or *fits*. A prefix `(target failed its control)` means the target itself is not trustworthy.
* **`n_clashes`, `worst_clash`**: heavy-atom pairs in the best-scoring pose closer than the sum of van der Waals radii minus 0.4 Å
  (hydrogen-bond N/O pairs: closer than 2.4 Å). `n_clash_free` of `n_poses` asks whether *any* docked pose avoids the protein.
* **`centroid_dist`, `site_contact_fraction`**: is the pose in the deposited ligand's site? A pose can be clash-free simply because it sits at the
  rim of the box. `site_contact_fraction` is the share of the residues touching the deposited ligand that also touch the pose (1.0 = same neighbourhood).
* **`best_score`** (kcal/mol, more negative = better): shown because Vina gives it, **not a ranking**. In the SidA tests, with a 16-22 Å box a large
  hydrophobic molecule scored better than the real substrate because it could sit at the rim; and in our earlier 22 Å analysis of 4B63 the deposited
  ornithine pose scored about 0.4 kcal/mol *worse* than a wrong pose. Compare clashes and site occupancy, not scores.
* **The box matters.** Re-run with `box_size=22` to see an oversized ligand stop clashing and move to the rim; that is the box, not a fit.
  Conversely, a ligand that really extends beyond the crystal ligand's site will be penalised by a 12 Å box: compare box sizes before concluding.
* **Protonation matters.** SidA's δ-amine and the ligand pH are open questions; re-run with another `ligand_ph` to see whether a conclusion survives.

## 7. Look at a pose

Green = docked pose, grey sticks = the deposited ligand, red spheres = receptor atoms that clash.

In [ ]:
VIEW_TARGET = "6X0J_A"
VIEW_LIGAND = "tetraphenylmethane (too big on purpose)"

key = (VIEW_TARGET, VIEW_LIGAND)
S.show_pose(result.targets[VIEW_TARGET], result.poses[key], result.clashes[key]).show()

## 8. Clashes of the best pose, atom by atom

In [ ]:
rows = [{"target": t, "ligand": l, "clash": c.label()} for (t, l), cl in result.clashes.items() for c in cl[:5]]
pd.DataFrame(rows) if rows else print("no clashes in any best pose")

## 9. Compare with the reference run

`reference_results.csv` holds the table from the machine this notebook was built on (Linux, CPU Vina 1.2.7, the example parameters above, seeds 1-3).
If you changed any parameter above the comparison is meaningless. Vina is deterministic per seed on one machine; across machines small score differences are possible.

In [ ]:
ref_path = Path("reference_results.csv")
if ref_path.exists():
    ref = pd.read_csv(ref_path)
    both = result.results.merge(ref, on=["target", "ligand"], suffixes=("", "_ref"))
    both["score_diff"] = both["best_score"] - both["best_score_ref"]
    print(both[["target", "ligand", "best_score", "best_score_ref", "score_diff", "n_clashes", "n_clashes_ref"]].round(2).to_string(index=False))
    print("\nlargest |score difference|: %.2f kcal/mol" % both["score_diff"].abs().max())
else:
    print("no reference_results.csv next to the notebook")

## Observations  (experimenter's notes: write what you see)

*Date:*
*Targets that failed the control, and what that means for the rest:*
*Ligands that clash, where (residues), and in how many poses:*
*Ligands that are clash-free but outside the site:*
*Anything that differs from the reference run:*
*Next step:*

## Interpretation of the worked example (reference run, Linux, 12 Å box)

* **The control separates the structures.** The deposited ornithine is recovered in 6X0J chain A (median top-1 RMSD 0.97 Å, 5 of 5 seeds) and chain C (1.06 Å, 4 of 5 seeds), but **not in 4B63** (4.91 Å, 0 of 5 seeds).
  4B63 has the higher resolution of the two structures (1.90 Å against 2.335 Å) [7,11] and fails at every box size from 12 to 22 Å; 6X0J chain C fails when the box is 22 Å (5.96 Å) and passes at 12 Å. So the 4B63 readings in the table are about the model.
  The redocking of 4B63 is an open problem in our SidA benchmark, not a result about ornithine.
* **Ornithine, lysine and arginine fit.** In all three receptors each is clash-free and in the site (site-contact fraction 0.83-1.00). This is consistent with the crystal structures of SidA with lysine and arginine bound [7]. It says nothing about the enzyme's selectivity for ornithine [5,10].
* **The oversized molecule does not fit.** Tetraphenylmethane clashes in every docked pose (8-15 clashes in the reported pose, scores between +0.4 and +9.3 kcal/mol). With a 16-22 Å box it instead docks at the rim without clashes and scores better than the substrate,
  which is why the site-sized box and the `reading` column exist.
* **Scores are not a ranking here.** Vina scores arginine (-6.2 to -6.8 kcal/mol) better than lysine (-5.4 to -5.6) and lysine better than ornithine (-4.9 to -5.2) in all three receptors. This follows molecular size, as expected for interaction-energy scores [25], and does not follow the specificity of the enzyme [5,10].

## Limits

* **The control is a weak test in a tight, ligand-centred box** (see section 5). A target that fails it is a real warning; a target that passes has only shown the redock is possible.
* **Rigid receptor, one conformation per structure.** Induced fit, loop movement and the FAD in/out switch [11] are not modelled.
* **A clash is a geometric statement**, not a free energy. Thresholds (0.4 Å overlap, 2.4 Å for N/O pairs) were set from five deposited
  ornithine poses (worst non-polar overlap 0.16 Å, closest hydrogen bond 2.50 Å): a small calibration set.
* **Heavy atoms only**; hydrogens are not checked. Waters are removed, so water-mediated sites are not represented.
* **Only FAD, FDA and NADP can be kept as cofactors** (the only ones with a shipped template). Reduced NADPH (`NDP`), heme, FMN etc. are stripped and the list in section 3 says so.
* **Ligand preparation** takes the SMILES at face value after protonation; tautomers and ring conformers are not enumerated. The protonation state of ornithine's side-chain amine is debated [10]; test more than one `ligand_ph`.

## References

**Verification (2026-11-05).** The 22 journal references with a PubMed record (all except 20, 21 and 23) were matched to that record by DOI, and authors with initials, year, journal, volume, first page and title were compared (`tools/verify_notebook_references.py` repeats the check). Reference 23 (Bondi, 1964) is not indexed in PubMed and was checked in Crossref. References 20 and 21 are software web addresses (both reachable on the same date). Structure resolutions and space groups in the Methods table come from the RCSB data API.

1. Latgé, J.-P., and Chamilos, G. (2019) *Aspergillus fumigatus* and aspergillosis in 2019. *Clin. Microbiol. Rev.* 33, e00140-18. doi:10.1128/CMR.00140-18
2. Schrettl, M., Bignell, E., Kragl, C., Joechl, C., Rogers, T., Arst, H. N., Jr., Haynes, K., and Haas, H. (2004) Siderophore biosynthesis but not reductive iron assimilation is essential for *Aspergillus fumigatus* virulence. *J. Exp. Med.* 200, 1213-1219. doi:10.1084/jem.20041242
3. Hissen, A. H., Wan, A. N., Warwas, M. L., Pinto, L. J., and Moore, M. M. (2005) The *Aspergillus fumigatus* siderophore biosynthetic gene *sidA*, encoding L-ornithine N<sup>5</sup>-oxygenase, is required for virulence. *Infect. Immun.* 73, 5493-5503. doi:10.1128/IAI.73.9.5493-5503.2005
4. Haas, H. (2014) Fungal siderophore metabolism with a focus on *Aspergillus fumigatus*. *Nat. Prod. Rep.* 31, 1266-1276. doi:10.1039/c4np00071d
5. Chocklett, S. W., and Sobrado, P. (2010) *Aspergillus fumigatus* SidA is a highly specific ornithine hydroxylase with bound flavin cofactor. *Biochemistry* 49, 6777-6783. doi:10.1021/bi100291n
6. Huijbers, M. M., Montersino, S., Westphal, A. H., Tischler, D., and van Berkel, W. J. (2014) Flavin dependent monooxygenases. *Arch. Biochem. Biophys.* 544, 2-17. doi:10.1016/j.abb.2013.12.005
7. Franceschini, S., Fedkenheuer, M., Vogelaar, N. J., Robinson, H. H., Sobrado, P., and Mattevi, A. (2012) Structural insight into the mechanism of oxygen activation and substrate selectivity of flavin-dependent N-hydroxylating monooxygenases. *Biochemistry* 51, 7043-7045. doi:10.1021/bi301072w
8. Shirey, C., Badieyan, S., and Sobrado, P. (2013) Role of Ser-257 in the sliding mechanism of NADP(H) in the reaction catalyzed by the *Aspergillus fumigatus* flavin-dependent ornithine N<sup>5</sup>-monooxygenase SidA. *J. Biol. Chem.* 288, 32440-32448. doi:10.1074/jbc.M113.487181
9. Robinson, R., Qureshi, I. A., Klancher, C. A., Rodriguez, P. J., Tanner, J. J., and Sobrado, P. (2015) Contribution to catalysis of ornithine binding residues in ornithine N<sup>5</sup>-monooxygenase. *Arch. Biochem. Biophys.* 585, 25-31. doi:10.1016/j.abb.2015.09.008
10. Robinson, R. M., Klancher, C. A., Rodriguez, P. J., and Sobrado, P. (2019) Flavin oxidation in flavin-dependent N-monooxygenases. *Protein Sci.* 28, 90-99. doi:10.1002/pro.3487
11. Campbell, A. C., Stiers, K. M., Martin del Campo, J. S., Mehra-Chaudhary, R., Sobrado, P., and Tanner, J. J. (2020) Trapping conformational states of a flavin-dependent N-monooxygenase in crystallo reveals protein and flavin dynamics. *J. Biol. Chem.* 295, 13239-13249. doi:10.1074/jbc.RA120.014750
12. Pierdominici-Sottile, G., Palma, J., Ferrelli, M. L., and Sobrado, P. (2024) The dynamics of the flavin, NADPH, and active site loops determine the mechanism of activation of class B flavin-dependent monooxygenases. *Protein Sci.* 33, e4935. doi:10.1002/pro.4935
13. Berman, H. M., Westbrook, J., Feng, Z., Gilliland, G., Bhat, T. N., Weissig, H., Shindyalov, I. N., and Bourne, P. E. (2000) The Protein Data Bank. *Nucleic Acids Res.* 28, 235-242. doi:10.1093/nar/28.1.235
14. Dolinsky, T. J., Czodrowski, P., Li, H., Nielsen, J. E., Jensen, J. H., Klebe, G., and Baker, N. A. (2007) PDB2PQR: expanding and upgrading automated preparation of biomolecular structures for molecular simulations. *Nucleic Acids Res.* 35, W522-W525. doi:10.1093/nar/gkm276
15. Jurrus, E., Engel, D., Star, K., Monson, K., Brandi, J., Felberg, L. E., Brookes, D. H., Wilson, L., Chen, J., Liles, K., Chun, M., Li, P., *et al.* (2018) Improvements to the APBS biomolecular solvation software suite. *Protein Sci.* 27, 112-128. doi:10.1002/pro.3280
16. Olsson, M. H., Søndergaard, C. R., Rostkowski, M., and Jensen, J. H. (2011) PROPKA3: consistent treatment of internal and surface residues in empirical pK<sub>a</sub> predictions. *J. Chem. Theory Comput.* 7, 525-537. doi:10.1021/ct100578z
17. Trott, O., and Olson, A. J. (2010) AutoDock Vina: improving the speed and accuracy of docking with a new scoring function, efficient optimization, and multithreading. *J. Comput. Chem.* 31, 455-461. doi:10.1002/jcc.21334
18. Eberhardt, J., Santos-Martins, D., Tillack, A. F., and Forli, S. (2021) AutoDock Vina 1.2.0: new docking methods, expanded force field, and Python bindings. *J. Chem. Inf. Model.* 61, 3891-3898. doi:10.1021/acs.jcim.1c00203
19. O'Boyle, N. M., Banck, M., James, C. A., Morley, C., Vandermeersch, T., and Hutchison, G. R. (2011) Open Babel: an open chemical toolbox. *J. Cheminform.* 3, 33. doi:10.1186/1758-2946-3-33
20. RDKit: open-source cheminformatics. https://www.rdkit.org (software; version 2026.03.1 used here).
21. Meeko: preparation of small molecules and receptors for AutoDock. Forli Lab, Scripps Research. https://github.com/forlilab/Meeko (software; version 0.8.0 used here).
22. Kunzmann, P., and Hamacher, K. (2018) Biotite: a unifying open source computational biology framework in Python. *BMC Bioinformatics* 19, 346. doi:10.1186/s12859-018-2367-z
23. Bondi, A. (1964) van der Waals volumes and radii. *J. Phys. Chem.* 68, 441-451. doi:10.1021/j100785a001
24. Chen, V. B., Arendall, W. B., III, Headd, J. J., Keedy, D. A., Immormino, R. M., Kapral, G. J., Murray, L. W., Richardson, J. S., and Richardson, D. C. (2010) MolProbity: all-atom structure validation for macromolecular crystallography. *Acta Crystallogr. D* 66, 12-21. doi:10.1107/S0907444909042073
25. Pan, Y., Huang, N., Cho, S., and MacKerell, A. D., Jr. (2003) Consideration of molecular weight during compound selection in virtual target-based database screening. *J. Chem. Inf. Comput. Sci.* 43, 267-272. doi:10.1021/ci020055f